# 📊 Baseline Model Evaluation & MLflow Experiment Tracking
**MLOps Course Project at RVCE** | Team: Samarth, Sambhav, Samiksha, Samith  
**Workstream**: Baseline Models & Evaluation Harness (Workstream D)  
**Reference Guidelines**: `docs/PROJECT_BRIEF.md` (Section 7.3), `AGENTS.md`, `docs/baseline_models.md`

---

## 🎯 Objectives
1. **Model Selection**: Benchmark probabilistic rainfall estimation against four fundamental baselines:
   - **B1**: Historical Climatological Mean / Median (Standard Civil Practice)
   - **B2**: Historical Empirical Quantiles ($P_{10}$ Dry, $P_{50}$ Normal, $P_{90}$ Wet)
   - **B3**: Lag-1 Autoregressive Persistence (Meteorological Benchmark)
   - **B4**: Linear Quantile Regression on Engineered Antecedent Features
2. **Strict Time-Series Evaluation**: Perform expanding-window backtesting ($1970 	o 2025$, 672 monthly predictions) with **zero lookahead leakage**.
3. **MLOps Tracking**: Log parameters, Pinball losses, MAE, coverage intervals, and prediction artifacts directly into **MLflow**.


In [1]:
import os
import sys

# Ensure execution from project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.append(os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

print(f"Active Root Directory: {os.getcwd()}")
print(f"MLflow Version: {mlflow.__version__}")


2026/11/01 01:13:23 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Users\user\anaconda3\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Active Root Directory: C:\Users\user\Desktop\mlops_new
MLflow Version: 3.17.2


---
## 1. Load Processed Feature Dataset
We load `data/processed/features_bengaluru_monthly.csv` tracked via DVC.
Features were generated strictly using prior information ($\le t-1$).


In [2]:
df_monthly = pd.read_csv('data/processed/features_bengaluru_monthly.csv')
print(f"Monthly Feature Dataset Shape: {df_monthly.shape}")
print(f"Years: {df_monthly['year'].min()} to {df_monthly['year'].max()}")

feature_cols = [
    'month_sin', 'month_cos', 
    'rain_lag1m', 'rain_lag2m', 
    'wet_days_lag1m', 
    'rolling_3m_sum', 'rolling_12m_sum', 
    'departure_pct_lag1m'
]
print("Pre-selected Features for Quantile Regression:")
for col in feature_cols:
    print(f"  • {col}")

df_monthly[['year', 'month', 'monthly_rain_mm'] + feature_cols].head(5)


Monthly Feature Dataset Shape: (1500, 16)
Years: 1901 to 2025
Pre-selected Features for Quantile Regression:
  • month_sin
  • month_cos
  • rain_lag1m
  • rain_lag2m
  • wet_days_lag1m
  • rolling_3m_sum
  • rolling_12m_sum
  • departure_pct_lag1m


,year,month,monthly_rain_mm,month_sin,month_cos,rain_lag1m,rain_lag2m,wet_days_lag1m,rolling_3m_sum,rolling_12m_sum,departure_pct_lag1m
0,1901,1,0.000000,0.500000,8.660254e-01,NaN,NaN,NaN,NaN,NaN,NaN
1,1901,2,39.847523,0.866025,5.000000e-01,0.000000,NaN,0.0,0.000000,NaN,-99.997677
2,1901,3,0.000000,1.000000,6.123234e-17,39.847523,0.000000,6.0,39.847523,NaN,517.442151
3,1901,4,29.297655,0.866025,-5.000000e-01,0.000000,39.847523,0.0,39.847523,NaN,-99.999128
4,1901,5,156.685188,0.500000,-8.660254e-01,29.297655,0.000000,8.0,69.145178,NaN,-30.579305


---
## 2. Execute Expanding-Window Backtesting (1970 – 2025)
Per `AGENTS.md`:
- Train on years $< T$ (minimum 30 years warm-up).
- Predict on out-of-sample year $T$.
- Repeat across 56 test years ($1970 	o 2025$).


In [3]:
from src.evaluation.backtest import ExpandingWindowEvaluator, log_evaluation_to_mlflow

evaluator = ExpandingWindowEvaluator(
    min_train_years=30,
    eval_start_year=1970,
    eval_end_year=2025,
    quantiles=[0.10, 0.50, 0.90]
)

print("Starting expanding-window backtest on monthly features...")
pred_df, metrics = evaluator.run_monthly_evaluation(
    df_monthly, 
    feature_cols=feature_cols,
    target_col='monthly_rain_mm',
    group_col='month'
)
print(f"Backtest completed! Evaluated {len(pred_df)} test months across 1970-2025.")


Starting expanding-window backtest on monthly features...
Backtest completed! Evaluated 672 test months across 1970-2025.


---
## 3. Benchmark Metrics Comparison Table
Let's aggregate the metrics to compare point-estimate accuracy (MAE) and probabilistic calibration (Pinball Loss & 80% Coverage).


In [4]:
summary_rows = [
    {
        "Model ID": "B1 (Mean)",
        "Model Name": "Climatological Mean",
        "MAE (mm)": f"{metrics['B1_climatology_mean']['mae']:.2f}",
        "Pinball Q10": "—",
        "Pinball Q50": "—",
        "Pinball Q90": "—",
        "80% Interval Coverage": "—"
    },
    {
        "Model ID": "B1 (Median)",
        "Model Name": "Climatological Median",
        "MAE (mm)": f"{metrics['B1_climatology_median']['mae']:.2f}",
        "Pinball Q10": "—",
        "Pinball Q50": "—",
        "Pinball Q90": "—",
        "80% Interval Coverage": "—"
    },
    {
        "Model ID": "B2",
        "Model Name": "Empirical Quantiles (P10, P50, P90)",
        "MAE (mm)": f"{metrics['B2_empirical_quantiles']['mae_median']:.2f}",
        "Pinball Q10": f"{metrics['B2_empirical_quantiles']['pinball_loss_q10']:.2f}",
        "Pinball Q50": f"{metrics['B2_empirical_quantiles']['pinball_loss_q50']:.2f}",
        "Pinball Q90": f"{metrics['B2_empirical_quantiles']['pinball_loss_q90']:.2f}",
        "80% Interval Coverage": f"{metrics['B2_empirical_quantiles']['coverage_80pct']:.2f}%"
    },
    {
        "Model ID": "B3",
        "Model Name": "Lag-1 Persistence",
        "MAE (mm)": f"{metrics['B3_persistence_lag1']['mae']:.2f}",
        "Pinball Q10": "—",
        "Pinball Q50": "—",
        "Pinball Q90": "—",
        "80% Interval Coverage": "—"
    },
    {
        "Model ID": "B4",
        "Model Name": "Quantile Regression (Engineered Features)",
        "MAE (mm)": f"{metrics['B4_quantile_regression']['mae_median']:.2f}",
        "Pinball Q10": f"{metrics['B4_quantile_regression']['pinball_loss_q10']:.2f}",
        "Pinball Q50": f"{metrics['B4_quantile_regression']['pinball_loss_q50']:.2f}",
        "Pinball Q90": f"{metrics['B4_quantile_regression']['pinball_loss_q90']:.2f}",
        "80% Interval Coverage": f"{metrics['B4_quantile_regression']['coverage_80pct']:.2f}%"
    }
]

metrics_df = pd.DataFrame(summary_rows)
display(metrics_df)


,Model ID,Model Name,MAE (mm),Pinball Q10,Pinball Q50,Pinball Q90,80% Interval Coverage
0,B1 (Mean),Climatological Mean,34.57,—,—,—,—
1,B1 (Median),Climatological Median,33.81,—,—,—,—
2,B2,"Empirical Quantiles (P10, P50, P90)",33.81,5.85,16.90,11.00,80.95%
3,B3,Lag-1 Persistence,48.13,—,—,—,—
4,B4,Quantile Regression (Engineered Features),40.83,6.14,20.42,12.45,79.76%


---
## 4. Visualizing Model Predictions vs Actual Rainfall
Let's plot a recent 10-year window (2015 – 2025) to visually inspect:
1. Actual observations vs Normal ($P_{50}$) and Dry ($P_{10}$) bounds.
2. How well the Empirical Quantile envelope ($P_{10} 	o P_{90}$) encompasses extreme rainfall years.


In [5]:
recent_df = pred_df[pred_df['year'] >= 2015].copy()
recent_df['date_label'] = recent_df['year'].astype(str) + '-' + recent_df['month'].astype(str).str.zfill(2)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Subplot 1: Empirical Quantile Envelopes (B2)
ax1.plot(range(len(recent_df)), recent_df['actual'], color='black', linewidth=2, label='Actual Monthly Rainfall (mm)', marker='o', markersize=3)
ax1.plot(range(len(recent_df)), recent_df['b2_q50'], color='blue', linestyle='--', linewidth=1.8, label='B2: Normal Year Median (P50)')
ax1.fill_between(range(len(recent_df)), recent_df['b2_q10'], recent_df['b2_q90'], color='skyblue', alpha=0.35, label='B2: 80% Uncertainty Band [P10 - P90]')
ax1.plot(range(len(recent_df)), recent_df['b2_q10'], color='crimson', linestyle=':', linewidth=1.5, label='B2: Drought Year Bound (P10)')
ax1.set_title('Figure 1A: Empirical Climatological Quantile Envelope (B2) vs Actuals (2015 - 2025)', fontsize=13, fontweight='bold')
ax1.set_ylabel('Rainfall (mm)', fontsize=11)
ax1.legend(loc='upper right', frameon=True)

# Subplot 2: Quantile Regression on Features (B4)
ax2.plot(range(len(recent_df)), recent_df['actual'], color='black', linewidth=2, label='Actual Monthly Rainfall (mm)', marker='o', markersize=3)
ax2.plot(range(len(recent_df)), recent_df['b4_q50'], color='green', linestyle='--', linewidth=1.8, label='B4: Regression Median (P50)')
ax2.fill_between(range(len(recent_df)), recent_df['b4_q10'], recent_df['b4_q90'], color='lightgreen', alpha=0.35, label='B4: 80% Uncertainty Band [P10 - P90]')
ax2.plot(range(len(recent_df)), recent_df['b4_q10'], color='darkorange', linestyle=':', linewidth=1.5, label='B4: Drought Year Bound (P10)')
ax2.set_title('Figure 1B: Dynamic Quantile Regression Envelope (B4) vs Actuals (2015 - 2025)', fontsize=13, fontweight='bold')
ax2.set_ylabel('Rainfall (mm)', fontsize=11)
ax2.set_xlabel('Time (Months)', fontsize=11)

# Set x-ticks every 12 months (annual markers)
tick_indices = range(0, len(recent_df), 12)
tick_labels = [recent_df.iloc[i]['year'] for i in tick_indices]
ax2.set_xticks(tick_indices)
ax2.set_xticklabels(tick_labels, fontsize=11, fontweight='bold')
ax2.legend(loc='upper right', frameon=True)

plt.tight_layout()
plt.show()


<Figure size 1600x1000 with 2 Axes>

---
## 5. MLOps Experiment Tracking: Logging to MLflow
We log parameters, Pinball loss, and out-of-sample backtest CSV artifacts to the MLflow tracking registry (`sqlite:///mlflow.db`).


In [6]:
log_evaluation_to_mlflow(metrics, pred_df, tracking_uri='sqlite:///mlflow.db')

# Verify runs recorded in MLflow
client = mlflow.tracking.MlflowClient(tracking_uri='sqlite:///mlflow.db')
experiment = client.get_experiment_by_name("rainfall_baseline_evaluation")
runs = client.search_runs(experiment_ids=[experiment.experiment_id])

print(f"\nActive MLflow Experiment: '{experiment.name}' (ID: {experiment.experiment_id})")
print(f"Total Model Runs Logged: {len(runs)}")
run_summary = []
for r in runs:
    run_summary.append({
        "Run Name": r.data.tags.get("mlflow.runName", "Unnamed"),
        "Status": r.info.status,
        "Run ID": r.info.run_id[:8],
        "Logged Metrics Count": len(r.data.metrics)
    })
display(pd.DataFrame(run_summary))


Logged B1_climatology_mean to MLflow successfully.
Logged B1_climatology_median to MLflow successfully.
Logged B2_empirical_quantiles to MLflow successfully.
Logged B3_persistence_lag1 to MLflow successfully.
Logged B4_quantile_regression to MLflow successfully.

Active MLflow Experiment: 'rainfall_baseline_evaluation' (ID: 1)
Total Model Runs Logged: 20


,Run Name,Status,Run ID,Logged Metrics Count
0,B4_quantile_regression,FINISHED,6f1e712f,5
1,B3_persistence_lag1,FINISHED,5f128edc,1
2,B2_empirical_quantiles,FINISHED,95677b14,5
3,B1_climatology_median,FINISHED,319c47cc,1
4,B1_climatology_mean,FINISHED,f5d3c8c4,1
5,B4_quantile_regression,FINISHED,f1452f8b,5
6,B3_persistence_lag1,FINISHED,8b94f5f6,1
7,B2_empirical_quantiles,FINISHED,7204c3e1,5
8,B1_climatology_median,FINISHED,c2d46784,1
9,B1_climatology_mean,FINISHED,02b2eed3,1


---
## 🏁 Scientific Conclusions & Next Pipeline Steps
1. **Empirical Quantiles (B2) is the Strongest Distributional Baseline**:
   - Achieves exact nominal coverage ($80.95\%$ for target $80\%$).
   - Gives the lowest Pinball Loss ($5.85$ for $Q_{10}$, $16.90$ for $Q_{50}$).
2. **Persistence (B3) Fails Heavily ($MAE = 48.13	ext{ mm}$)**:
   - Intra-annual monsoonal oscillations in Bengaluru possess low autoregressive persistence across years.
3. **Linear Quantile Regression (B4)**:
   - Linear combinations of antecedent features do not beat static climatology without non-linear interactions.
   - **Criterion for Advanced ML Models**: Future models (e.g. LightGBM Quantile or Bayesian Hierarchical models) must achieve Pinball Loss $< 5.85$ on $Q_{10}$ and $MAE < 33.81	ext{ mm}$ to prove real value over B2!
4. **Next Phase Milestone**:
   - Implement deterministic hydrology balance engine (`src/hydrology/`).
   - Leave-one-gauge-out spatial cross-validation for KSNDMC micro-climate adjustments.
